# Dataset Load
- 의료보험 데이터를 활용하여, 피보험자의 조건에 따라 보험료가 책정될지 예측하는 Regression 모델 생성하기
- 이전 모델(ver0)에서는, R^2가 0.81에 불과하였다. 특히 보험료가 높게 측정된 사람에 대한 이상치에 대한 예측이 좋지 않았다. 이를 0.9 이상으로 고도화하고자 한다.
- 우선, 이 버전의 모델에서는 타겟 변수 (changes)를 **로그 변환하여 정규분포에 가깝게 만든 뒤 학습**시킬 것이다. 이후, 다시 단위를 복원하여 극단적인 이상치에 대한 예측력을 높이고자 한다.
- Column 설명
    - Age : 나이
    - Sex : 성별
    - BMI : 체질량지수
    - Children : 자녀수
    - Smoker : 흡연여부
    - Region : 거주지역
    - Charges : 보험료

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

data_path = '/content/drive/MyDrive/Colab Notebooks/PyTorch 실습 /insurance.csv'

df = pd.read_csv(data_path)

# Data 기초 분석

## Data 행, 열 개수 확인

In [4]:
df.shape

(1338, 7)

## Data 상위 5개열 확인

In [5]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Data 기초 통계정보 확인

In [6]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## Data 카테고리 데이터 column별 분포 확인

In [7]:
for col in df.select_dtypes("object").columns:
    print(df[col].value_counts())
    print("===")

sex
male      676
female    662
Name: count, dtype: int64
===
smoker
no     1064
yes     274
Name: count, dtype: int64
===
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64
===


# Train Test Split

In [8]:
from sklearn.model_selection import train_test_split

x = df.copy().drop(columns = ['charges'])
y = df.copy()['charges']

x_train, x_test, y_train, y_test =  train_test_split(x, y, test_size = 0.2, random_state = 2024)
x_train, x_val, y_train, y_val =  train_test_split(x_train, y_train, test_size = 0.2, random_state = 2024)

In [9]:
print(x_train.shape, x_test.shape, x_val.shape)

(856, 6) (268, 6) (214, 6)


# Label Encoding

In [7]:
label_cols = ['sex', 'smoker']

from sklearn.preprocessing import LabelEncoder
from collections import defaultdict


class MultiLabelEncoder:
    def __init__(self):
        self.label_encoders = defaultdict(LabelEncoder)

    def fit(self, X, columns):
        for column in columns:
            self.label_encoders[column].fit(X[column])
        return self

    def transform(self, X, columns):
        X_transformed = X.copy()
        for column in columns:
            X_transformed[column] = self.label_encoders[column].transform(X[column])
        return X_transformed

    def fit_transform(self, X, columns):
        self.fit(X, columns)
        return self.transform(X, columns)

    def inverse_transform(self, X, columns):
        X_inverse_transformed = X.copy()
        for column in columns:
            X_inverse_transformed[column] = self.label_encoders[column].inverse_transform(X[column])
        return X_inverse_transformed

In [11]:
multilabelencoder = MultiLabelEncoder()
multilabelencoder.fit(x_train, label_cols)
x_train = multilabelencoder.transform(x_train, label_cols)
x_test = multilabelencoder.transform(x_test, label_cols)
x_val = multilabelencoder.transform(x_val, label_cols)

# One Hot Encoding

In [9]:
from sklearn.preprocessing import OneHotEncoder


class DataFrameOneHotEncoder:
    def __init__(self):
        self.encoder = OneHotEncoder(sparse_output=False)
        self.columns = None
        self.feature_names = None

    def fit(self, X, columns):
        self.columns = columns
        self.encoder.fit(X[columns])
        self.feature_names = self.encoder.get_feature_names_out(columns)
        return self

    def transform(self, X):
        X_transformed = X.drop(columns=self.columns).reset_index(drop=True)
        encoded_columns = pd.DataFrame(self.encoder.transform(X[self.columns]), columns=self.feature_names)
        X_transformed = pd.concat([X_transformed, encoded_columns], axis=1)
        return X_transformed

    def fit_transform(self, X, columns):
        self.fit(X, columns)
        return self.transform(X)

    def inverse_transform(self, X):
        original_columns = pd.DataFrame(self.encoder.inverse_transform(X[self.feature_names]), columns=self.columns)
        X_original = X.drop(columns=self.feature_names).reset_index(drop=True)
        X_original = pd.concat([X_original, original_columns], axis=1)
        return X_original


In [10]:
oe_cols = ['region']

onehotencoder = DataFrameOneHotEncoder()
onehotencoder.fit(x_train, oe_cols)
x_train = onehotencoder.transform(x_train.copy())
x_test = onehotencoder.transform(x_test.copy())
x_val = onehotencoder.transform(x_val.copy())

# Scaling

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = list(set(df.select_dtypes(exclude = "object").columns ) - {'charges'})

scaler.fit(x_train[numeric_cols])
x_train[numeric_cols] = scaler.transform(x_train[numeric_cols].copy())
x_test[numeric_cols] = scaler.transform(x_test[numeric_cols].copy())
x_val[numeric_cols] = scaler.transform(x_val[numeric_cols].copy())

In [15]:
x_train.head()

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
0,-1.505337,0,1.127546,-0.102112,0,0.0,0.0,1.0,0.0
1,1.608002,0,-0.901425,-0.926717,0,0.0,0.0,0.0,1.0
2,1.254213,0,-0.107336,-0.926717,1,0.0,0.0,1.0,0.0
3,1.041940,0,2.681055,0.722492,0,0.0,0.0,0.0,1.0
4,-1.222306,0,-0.397897,-0.926717,0,0.0,0.0,1.0,0.0


## PyTorch Modeling

## 1. Library Import

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

## 2. Dataset / Data Loader 생성 / 선언

In [17]:
# 1. 타겟 변수(Charges)에 로그 변환 적용
# np.log1p는 np.log(1 + x)와 같으며 값이 0일 때의 오류를 방지합니다.
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

# 2. Pandas DataFrame/Series를 PyTorch Tensor로 변환
# (이전 스케일링 단계의 결과물 형태에 따라 .values가 필요할 수 있습니다)
if isinstance(x_train, pd.DataFrame):
    x_train_tensor = torch.Tensor(x_train.values)
    x_val_tensor = torch.Tensor(x_val.values)
    x_test_tensor = torch.Tensor(x_test.values)
else:
    x_train_tensor = torch.Tensor(x_train)
    x_val_tensor = torch.Tensor(x_val)
    x_test_tensor = torch.Tensor(x_test)

# 로그 변환된 y값을 Tensor로 변환하고 (N, 1) 차원으로 맞춤
y_train_tensor = torch.Tensor(y_train_log.values).unsqueeze(1)
y_val_tensor = torch.Tensor(y_val_log.values).unsqueeze(1)
y_test_tensor = torch.Tensor(y_test_log.values).unsqueeze(1)

# 3. TensorDataset 및 DataLoader 생성
batch_size = 32

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 3. Model Class 생성 / 선언

In [18]:
class InsuranceRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(InsuranceRegressionModel, self).__init__()
        # 입력층 -> 은닉층 -> 출력층 구조
        self.linear_1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(64, 32)
        self.linear_3 = nn.Linear(32, 16)
        self.linear_4 = nn.Linear(16, 1) # 회귀 예측이므로 출력 노드는 1개

    def forward(self, x):
        x = self.linear_1(x)
        x = self.relu(x)
        x = self.linear_2(x)
        x = self.relu(x)
        x = self.linear_3(x)
        x = self.relu(x)
        x = self.linear_4(x)
        # 회귀는 최종 활성화 함수(Softmax 등) 없이 값을 그대로 반환
        return x

# 디바이스 설정 및 모델 선언
device = "cuda" if torch.cuda.is_available() else "cpu"
input_dim = x_train.shape[1]
model = InsuranceRegressionModel(input_dim).to(device)

## 4. Optimizer, Loss 선언

In [19]:
# 학습률 설정 및 Adam 옵티마이저 사용
optimizer = optim.Adam(params=model.parameters(), lr=0.01)

# 회귀 모델 손실 함수: 평균 제곱 오차(MSE)
loss_fn = nn.MSELoss().to(device)

## 5. Model Train

In [20]:
epochs = 200

for epoch in range(epochs):
    # Train Phase
    model.train()
    train_cost = 0
    train_batch_length = len(train_dataloader)

    for train_inputs, train_labels in train_dataloader:
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        # 모델 예측 및 손실 계산
        train_preds = model(train_inputs)
        train_loss = loss_fn(train_preds, train_labels)

        # 가중치 업데이트 (Backpropagation)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        train_cost += train_loss.item()

    train_cost /= train_batch_length

    # Validation Phase
    model.eval()
    val_cost = 0
    val_batch_length = len(val_dataloader)

    with torch.no_grad():
        for val_inputs, val_labels in val_dataloader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            val_preds = model(val_inputs)
            val_loss = loss_fn(val_preds, val_labels)

            val_cost += val_loss.item()

    val_cost /= val_batch_length

    # 10 Epoch 단위로 학습 경과 출력
    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}/{epochs} => Train Loss(MSE): {train_cost:.4f}, Val Loss(MSE): {val_cost:.4f}")

Epoch: 10/200 => Train Loss(MSE): 0.1678, Val Loss(MSE): 0.1692
Epoch: 20/200 => Train Loss(MSE): 0.1533, Val Loss(MSE): 0.1340
Epoch: 30/200 => Train Loss(MSE): 0.1462, Val Loss(MSE): 0.1385
Epoch: 40/200 => Train Loss(MSE): 0.1514, Val Loss(MSE): 0.1699
Epoch: 50/200 => Train Loss(MSE): 0.1590, Val Loss(MSE): 0.1394
Epoch: 60/200 => Train Loss(MSE): 0.1459, Val Loss(MSE): 0.2027
Epoch: 70/200 => Train Loss(MSE): 0.1568, Val Loss(MSE): 0.1665
Epoch: 80/200 => Train Loss(MSE): 0.1417, Val Loss(MSE): 0.1554
Epoch: 90/200 => Train Loss(MSE): 0.1897, Val Loss(MSE): 0.1987
Epoch: 100/200 => Train Loss(MSE): 0.1287, Val Loss(MSE): 0.2186
Epoch: 110/200 => Train Loss(MSE): 0.1458, Val Loss(MSE): 0.1409
Epoch: 120/200 => Train Loss(MSE): 0.1309, Val Loss(MSE): 0.1333
Epoch: 130/200 => Train Loss(MSE): 0.1554, Val Loss(MSE): 0.1715
Epoch: 140/200 => Train Loss(MSE): 0.1398, Val Loss(MSE): 0.1452
Epoch: 150/200 => Train Loss(MSE): 0.1760, Val Loss(MSE): 0.2107
Epoch: 160/200 => Train Loss(MSE):

## 6. Evaluation

In [21]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 모델 평가 모드
model.eval()

with torch.no_grad():
    # 테스트 데이터 예측 (로그 스케일의 예측값)
    test_preds_log = model(x_test_tensor.to(device))
    test_preds_log = test_preds_log.cpu().numpy()

    # 정답 데이터 (로그 스케일)
    test_trues_log = y_test_tensor.numpy()

# 로그 스케일로 예측된 값과 정답을 원래 스케일(달러)로 복원 (np.expm1)
test_preds_restored = np.expm1(test_preds_log)
test_trues_restored = np.expm1(test_trues_log)

# 복원된 값으로 회귀 평가 지표 계산
mse = mean_squared_error(test_trues_restored, test_preds_restored)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_trues_restored, test_preds_restored)
r2 = r2_score(test_trues_restored, test_preds_restored)

print("=== 로그 변환 적용 후 의료보험료 예측 평가 결과 ===")
print(f"MSE (평균 제곱 오차): {mse:.2f}")
print(f"RMSE (평균 제곱근 오차): {rmse:.2f}")
print(f"MAE (평균 절대 오차): {mae:.2f}")
print(f"R2 Score (결정계수): {r2:.4f}")

=== 로그 변환 적용 후 의료보험료 예측 평가 결과 ===
MSE (평균 제곱 오차): 43654008.00
RMSE (평균 제곱근 오차): 6607.12
MAE (평균 절대 오차): 3362.96
R2 Score (결정계수): 0.7081


와우! 오히려 성능이 떨어졌다!!!

1. 지수 변환(exp)의 폭발성과 MSE의 함정딥러닝 모델은 이제 '수만 달러'가 아닌 '6~11 사이의 로그값'을 맞추도록 학습되었다.
2. 예시: 로그 스케일에서 모델이 10.0을 예측해야 하는데 10.5를 예측했다고 가정해 보자. 로그 스케일에서는 오차가 고작 0.5. 하지만 이를 다시 달러로 복원(np.expm1)하면 어떻게 될까?
3. 정답: $e^{10} \approx 22,026$ 달러예측: $e^{10.5} \approx 36,315$ 달러원본 스케일에서는 단번에 14,000달러라는 엄청난 오차로 뻥튀기된다!
4. $R^2$와 MSE 지표는 '큰 오차'에 매우 강력한 페널티를 주기 때문에, 로그 스케일에서의 미세한 오차가 복원 후 전체 점수를 깎아먹은 것.

#4.2. Optimizer, Loss 선언 (학습률 조정)

In [22]:
# 학습률(lr)을 0.001로 대폭 낮추어 안정적으로 수렴하도록 유도
optimizer = optim.Adam(params=model.parameters(), lr=0.001)

# 회귀 모델 손실 함수: 평균 제곱 오차(MSE)
loss_fn = nn.MSELoss().to(device)

#5. Model Train (에포크 증가)

In [23]:
# 에포크를 200에서 500으로 증가
epochs = 500

for epoch in range(epochs):
    # Train Phase
    model.train()
    train_cost = 0
    train_batch_length = len(train_dataloader)

    for train_inputs, train_labels in train_dataloader:
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        # 모델 예측 및 손실 계산
        train_preds = model(train_inputs)
        train_loss = loss_fn(train_preds, train_labels)

        # 가중치 업데이트 (Backpropagation)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        train_cost += train_loss.item()

    train_cost /= train_batch_length

    # Validation Phase
    model.eval()
    val_cost = 0
    val_batch_length = len(val_dataloader)

    with torch.no_grad():
        for val_inputs, val_labels in val_dataloader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            val_preds = model(val_inputs)
            val_loss = loss_fn(val_preds, val_labels)

            val_cost += val_loss.item()

    val_cost /= val_batch_length

    # 에포크가 길어졌으므로 50 단위로 경과를 출력합니다.
    if (epoch + 1) % 50 == 0:
        print(f"Epoch: {epoch+1}/{epochs} => Train Loss(MSE): {train_cost:.4f}, Val Loss(MSE): {val_cost:.4f}")

Epoch: 50/500 => Train Loss(MSE): 0.0990, Val Loss(MSE): 0.1456
Epoch: 100/500 => Train Loss(MSE): 0.0960, Val Loss(MSE): 0.1510
Epoch: 150/500 => Train Loss(MSE): 0.0964, Val Loss(MSE): 0.1519
Epoch: 200/500 => Train Loss(MSE): 0.0960, Val Loss(MSE): 0.1528
Epoch: 250/500 => Train Loss(MSE): 0.0879, Val Loss(MSE): 0.1584
Epoch: 300/500 => Train Loss(MSE): 0.0845, Val Loss(MSE): 0.1598
Epoch: 350/500 => Train Loss(MSE): 0.0797, Val Loss(MSE): 0.1695
Epoch: 400/500 => Train Loss(MSE): 0.0781, Val Loss(MSE): 0.1705
Epoch: 450/500 => Train Loss(MSE): 0.0757, Val Loss(MSE): 0.1767
Epoch: 500/500 => Train Loss(MSE): 0.0741, Val Loss(MSE): 0.1799


#6.2 Evaluation

In [24]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 모델 평가 모드
model.eval()

with torch.no_grad():
    # 테스트 데이터 예측 (로그 스케일의 예측값)
    test_preds_log = model(x_test_tensor.to(device))
    test_preds_log = test_preds_log.cpu().numpy()

    # 정답 데이터 (로그 스케일)
    test_trues_log = y_test_tensor.numpy()

# 로그 스케일로 예측된 값과 정답을 원래 스케일(달러)로 복원 (np.expm1)
test_preds_restored = np.expm1(test_preds_log)
test_trues_restored = np.expm1(test_trues_log)

# 복원된 값으로 회귀 평가 지표 계산
mse = mean_squared_error(test_trues_restored, test_preds_restored)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_trues_restored, test_preds_restored)
r2 = r2_score(test_trues_restored, test_preds_restored)

print("=== 로그 변환 적용 후 의료보험료 예측 평가 결과 ===")
print(f"MSE (평균 제곱 오차): {mse:.2f}")
print(f"RMSE (평균 제곱근 오차): {rmse:.2f}")
print(f"MAE (평균 절대 오차): {mae:.2f}")
print(f"R2 Score (결정계수): {r2:.4f}")

=== 로그 변환 적용 후 의료보험료 예측 평가 결과 ===
MSE (평균 제곱 오차): 47932884.00
RMSE (평균 제곱근 오차): 6923.36
MAE (평균 절대 오차): 3572.13
R2 Score (결정계수): 0.6795


그냥 자동으로 모델 구조와 하이퍼파라미터를 최적화하자.. (optuna)

In [25]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.0 MB/s eta 0:00:00


In [26]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Optuna 전용 유연한 모델 클래스 정의
class OptunaInsuranceModel(nn.Module):
    def __init__(self, trial, input_dim):
        super(OptunaInsuranceModel, self).__init__()

        # 튜닝할 파라미터 (은닉층 노드 수를 Optuna가 선택하게 함)
        n_units_l1 = trial.suggest_int("n_units_l1", 16, 128)
        n_units_l2 = trial.suggest_int("n_units_l2", 16, 128)
        n_units_l3 = trial.suggest_int("n_units_l3", 8, 64)

        self.linear_1 = nn.Linear(input_dim, n_units_l1)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(n_units_l1, n_units_l2)
        self.linear_3 = nn.Linear(n_units_l2, n_units_l3)
        self.linear_4 = nn.Linear(n_units_l3, 1)

    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.relu(self.linear_3(x))
        return self.linear_4(x)

# 2. 목적 함수(Objective Function) 정의
def objective(trial):
    # Optuna가 탐색할 하이퍼파라미터 공간 정의
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True) # 학습률 탐색 (0.0001 ~ 0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    epochs = 200 # 탐색 속도를 위해 200으로 고정

    # DataLoader 재설정 (배치 사이즈가 바뀌므로)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # 모델 및 옵티마이저 선언
    model = OptunaInsuranceModel(trial, input_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    # 모델 학습
    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(x_batch)
            loss = loss_fn(preds, y_batch)
            loss.backward()
            optimizer.step()

    # 검증 (복원된 최종 MSE 기준으로 성능 평가)
    model.eval()
    val_preds_list = []
    val_trues_list = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            preds = model(x_batch.to(device)).cpu().numpy()
            val_preds_list.extend(preds)
            val_trues_list.extend(y_batch.numpy())

    # 지수 변환으로 원래 스케일 복원
    val_preds_restored = np.expm1(val_preds_list)
    val_trues_restored = np.expm1(val_trues_list)

    # 복원된 값의 MSE 계산
    mse = mean_squared_error(val_trues_restored, val_preds_restored)
    return mse # Optuna는 이 반환값(MSE)을 '최소화(Minimize)'하는 방향으로 학습함

# 3. Optuna Study 실행
print("🚀 하이퍼파라미터 자동 탐색을 시작합니다...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30) # 30번의 각기 다른 조합으로 테스트

print("\n🏆 최적의 하이퍼파라미터 조합:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-05-11 08:48:27,705] A new study created in memory with name: no-name-79f8cf2b-125e-495e-8103-17f339b2db1c


🚀 하이퍼파라미터 자동 탐색을 시작합니다...


[I 2026-05-11 08:48:36,098] Trial 0 finished with value: 30471412.0 and parameters: {'lr': 0.01362171826752809, 'batch_size': 64, 'n_units_l1': 99, 'n_units_l2': 28, 'n_units_l3': 64}. Best is trial 0 with value: 30471412.0.
[I 2026-05-11 08:48:56,193] Trial 1 finished with value: 32981520.0 and parameters: {'lr': 0.04115343171148768, 'batch_size': 16, 'n_units_l1': 122, 'n_units_l2': 88, 'n_units_l3': 52}. Best is trial 0 with value: 30471412.0.
[I 2026-05-11 08:49:06,917] Trial 2 finished with value: 86308784.0 and parameters: {'lr': 0.010908017130440463, 'batch_size': 32, 'n_units_l1': 102, 'n_units_l2': 112, 'n_units_l3': 27}. Best is trial 0 with value: 30471412.0.
[I 2026-05-11 08:49:17,806] Trial 3 finished with value: 34259304.0 and parameters: {'lr': 0.018799657867339743, 'batch_size': 32, 'n_units_l1': 102, 'n_units_l2': 89, 'n_units_l3': 46}. Best is trial 0 with value: 30471412.0.
[I 2026-05-11 08:49:37,907] Trial 4 finished with value: 38505504.0 and parameters: {'lr': 0.0


🏆 최적의 하이퍼파라미터 조합:
  lr: 0.004764199226757945
  batch_size: 64
  n_units_l1: 30
  n_units_l2: 76
  n_units_l3: 13


#1. DataLoader 재설정

In [5]:
batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

NameError: name 'DataLoader' is not defined

#2. 최적화된 노드 수로 Model Class 재선언

In [28]:
class OptimizedInsuranceModel(nn.Module):
    def __init__(self, input_dim):
        super(OptimizedInsuranceModel, self).__init__()
        # Optuna 최적화 노드 수 적용: 30 -> 76 -> 13
        self.linear_1 = nn.Linear(input_dim, 30)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(30, 76)
        self.linear_3 = nn.Linear(76, 13)
        self.linear_4 = nn.Linear(13, 1)

    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.relu(self.linear_3(x))
        return self.linear_4(x)

# 디바이스 설정 및 모델 선언
device = "cuda" if torch.cuda.is_available() else "cpu"
input_dim = x_train.shape[1]
model = OptimizedInsuranceModel(input_dim).to(device)

#3. 최적화된 학습률로 Optimizer 선언 및 학습




In [29]:
# 최적 학습률 적용
optimizer = optim.Adam(params=model.parameters(), lr=0.00476)
loss_fn = nn.MSELoss().to(device)

epochs = 500

print("🚀 최적화된 파라미터로 최종 학습을 시작합니다...")
for epoch in range(epochs):
    model.train()
    train_cost = 0

    for train_inputs, train_labels in train_dataloader:
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        optimizer.zero_grad()
        train_preds = model(train_inputs)
        train_loss = loss_fn(train_preds, train_labels)
        train_loss.backward()
        optimizer.step()

        train_cost += train_loss.item()

    train_cost /= len(train_dataloader)

    # 100 Epoch 단위로만 간략히 출력
    if (epoch + 1) % 100 == 0:
        print(f"Epoch: {epoch+1}/{epochs} => Train Loss(MSE): {train_cost:.4f}")

🚀 최적화된 파라미터로 최종 학습을 시작합니다...
Epoch: 100/500 => Train Loss(MSE): 0.1382
Epoch: 200/500 => Train Loss(MSE): 0.1180
Epoch: 300/500 => Train Loss(MSE): 0.1402
Epoch: 400/500 => Train Loss(MSE): 0.0935
Epoch: 500/500 => Train Loss(MSE): 0.1127


#4. 최종 Evaluation (복원 및 평가)

In [30]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model.eval()

with torch.no_grad():
    # 테스트 데이터 예측
    test_preds_log = model(x_test_tensor.to(device)).cpu().numpy()
    test_trues_log = y_test_tensor.numpy()

# np.expm1을 통해 원래 스케일(달러)로 복원
test_preds_restored = np.expm1(test_preds_log)
test_trues_restored = np.expm1(test_trues_log)

# 평가 지표 계산
mse = mean_squared_error(test_trues_restored, test_preds_restored)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_trues_restored, test_preds_restored)
r2 = r2_score(test_trues_restored, test_preds_restored)

print("\n=== 🏆 최적화 모델 최종 평가 결과 ===")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")


=== 🏆 최적화 모델 최종 평가 결과 ===
MSE: 34052940.00
RMSE: 5835.49
MAE: 3071.01
R2 Score: 0.7723


- 흡연과 비만이 동시에 있는 환자에서는 보험료가 폭발적으로 증가하는 패턴을 보인다.
- 따라서, **'흡연'과 '비만' 조건을 결합한 파생 변수 컬럼** (Feature Engineering)을 새로 추가하여, 딥러닝 모델이 더 쉽게 가중치를 찾도록 유도해보자
- 또한 지수 변환은 폐기한다.

#1. 파생변수 생성 및 데이터 전처리

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from collections import defaultdict

# ---------------------------------------------------------
# [A] 인코더 클래스 정의 (메모리 로딩)
# ---------------------------------------------------------
class MultiLabelEncoder:
    def __init__(self):
        self.label_encoders = defaultdict(LabelEncoder)

    def fit(self, X, columns):
        for column in columns:
            self.label_encoders[column].fit(X[column])
        return self

    def transform(self, X, columns):
        X_transformed = X.copy()
        for column in columns:
            X_transformed[column] = self.label_encoders[column].transform(X[column])
        return X_transformed

class DataFrameOneHotEncoder:
    def __init__(self):
        self.encoder = OneHotEncoder(sparse_output=False)
        self.columns = None
        self.feature_names = None

    def fit(self, X, columns):
        self.columns = columns
        self.encoder.fit(X[columns])
        self.feature_names = self.encoder.get_feature_names_out(columns)
        return self

    def transform(self, X):
        X_transformed = X.drop(columns=self.columns).reset_index(drop=True)
        encoded_columns = pd.DataFrame(self.encoder.transform(X[self.columns]), columns=self.feature_names)
        X_transformed = pd.concat([X_transformed, encoded_columns], axis=1)
        return X_transformed

# ---------------------------------------------------------
# [B] 데이터 로드 및 파생 변수 추가
# ---------------------------------------------------------
# df = pd.read_csv(data_path) # 이미 df가 메모리에 있다면 주석 처리 유지

# ✨ 핵심: 흡연자('yes')이면서 BMI가 30 이상인 경우 1, 아니면 0인 파생 변수 추가
df['high_risk'] = ((df['smoker'] == 'yes') & (df['bmi'] >= 30.0)).astype(int)
print("✅ 파생 변수 추가 완료! High Risk 환자 수:", df['high_risk'].sum())

# ---------------------------------------------------------
# [C] Train / Test Split (로그 변환 없는 원본 타겟)
# ---------------------------------------------------------
x = df.copy().drop(columns=['charges'])
y = df.copy()['charges']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2024)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=2024)

# ---------------------------------------------------------
# [D] 인코딩 (Label & One-Hot)
# ---------------------------------------------------------
label_cols = ['sex', 'smoker']
multilabelencoder = MultiLabelEncoder()
multilabelencoder.fit(x_train, label_cols)
x_train = multilabelencoder.transform(x_train, label_cols)
x_test = multilabelencoder.transform(x_test, label_cols)
x_val = multilabelencoder.transform(x_val, label_cols)

oe_cols = ['region']
onehotencoder = DataFrameOneHotEncoder()
onehotencoder.fit(x_train, oe_cols)
x_train = onehotencoder.transform(x_train.copy())
x_test = onehotencoder.transform(x_test.copy())
x_val = onehotencoder.transform(x_val.copy())

# ---------------------------------------------------------
# [E] 스케일링
# ---------------------------------------------------------
scaler = StandardScaler()
numeric_cols = list(set(x_train.select_dtypes(exclude="object").columns))

scaler.fit(x_train[numeric_cols])
x_train[numeric_cols] = scaler.transform(x_train[numeric_cols].copy())
x_test[numeric_cols] = scaler.transform(x_test[numeric_cols].copy())
x_val[numeric_cols] = scaler.transform(x_val[numeric_cols].copy())

print("✅ 모든 전처리 완료! 모델 학습을 진행할 준비가 되었습니다.")

✅ 파생 변수 추가 완료! High Risk 환자 수: 145
✅ 모든 전처리 완료! 모델 학습을 진행할 준비가 되었습니다.


#2. PyTorch Tensor 변환 및 DataLoader 생성

In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# DataFrame -> Tensor 변환
x_train_tensor = torch.Tensor(x_train.values)
x_val_tensor = torch.Tensor(x_val.values)
x_test_tensor = torch.Tensor(x_test.values)

# y값 텐서 변환 및 2차원(N, 1) 확장 (로그 변환 없음)
y_train_tensor = torch.Tensor(y_train.values).unsqueeze(1)
y_val_tensor = torch.Tensor(y_val.values).unsqueeze(1)
y_test_tensor = torch.Tensor(y_test.values).unsqueeze(1)

# Dataset 생성 (DataLoader는 Optuna 내부에서 배치 사이즈별로 새로 생성합니다)
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

input_dim = x_train.shape[1]
device = "cuda" if torch.cuda.is_available() else "cpu"

#3. Optuna 재실행

In [10]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.8 MB/s eta 0:00:00


In [11]:
import optuna
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error

class OptunaRawModel(nn.Module):
    def __init__(self, trial, input_dim):
        super(OptunaRawModel, self).__init__()
        n_units_l1 = trial.suggest_int("n_units_l1", 32, 128)
        n_units_l2 = trial.suggest_int("n_units_l2", 16, 64)
        n_units_l3 = trial.suggest_int("n_units_l3", 8, 32)

        self.linear_1 = nn.Linear(input_dim, n_units_l1)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(n_units_l1, n_units_l2)
        self.linear_3 = nn.Linear(n_units_l2, n_units_l3)
        self.linear_4 = nn.Linear(n_units_l3, 1)

    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.relu(self.linear_3(x))
        return self.linear_4(x)

def objective_raw(trial):
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True) # 학습률 탐색 범위 조정
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    epochs = 200

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = OptunaRawModel(trial, input_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    val_preds_list = []
    val_trues_list = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            preds = model(x_batch.to(device)).cpu().numpy()
            val_preds_list.extend(preds)
            val_trues_list.extend(y_batch.numpy())

    # 로그 변환이 없으므로 그대로 복원 없이 바로 MSE 계산
    mse = mean_squared_error(val_trues_list, val_preds_list)
    return mse

print("🚀 파생 변수 추가 모델 하이퍼파라미터 탐색 시작...")
study_raw = optuna.create_study(direction="minimize")
study_raw.optimize(objective_raw, n_trials=30)

print("\n🏆 최고 성능 하이퍼파라미터 조합:")
for key, value in study_raw.best_params.items():
    print(f"  {key}: {value}")

[I 2026-05-13 04:12:59,801] A new study created in memory with name: no-name-b7cb75e0-c5d5-4545-a35a-ccfd9b4b350b


🚀 파생 변수 추가 모델 하이퍼파라미터 탐색 시작...


[I 2026-05-13 04:13:29,097] Trial 0 finished with value: 19333596.57403718 and parameters: {'lr': 0.0020053711088220054, 'batch_size': 32, 'n_units_l1': 116, 'n_units_l2': 30, 'n_units_l3': 18}. Best is trial 0 with value: 19333596.57403718.
[I 2026-05-13 04:13:39,803] Trial 1 finished with value: 19200201.4744813 and parameters: {'lr': 0.014951731708246572, 'batch_size': 32, 'n_units_l1': 121, 'n_units_l2': 50, 'n_units_l3': 12}. Best is trial 1 with value: 19200201.4744813.
[I 2026-05-13 04:14:00,137] Trial 2 finished with value: 19129336.81405015 and parameters: {'lr': 0.006730734649562252, 'batch_size': 16, 'n_units_l1': 53, 'n_units_l2': 58, 'n_units_l3': 20}. Best is trial 2 with value: 19129336.81405015.
[I 2026-05-13 04:14:11,159] Trial 3 finished with value: 19744309.049098887 and parameters: {'lr': 0.03348266692598494, 'batch_size': 32, 'n_units_l1': 91, 'n_units_l2': 51, 'n_units_l3': 15}. Best is trial 2 with value: 19129336.81405015.
[I 2026-05-13 04:14:16,978] Trial 4 fin


🏆 최고 성능 하이퍼파라미터 조합:
  lr: 0.054140676564044296
  batch_size: 64
  n_units_l1: 44
  n_units_l2: 22
  n_units_l3: 19


#1. 최종 모델 아키텍처 및 하이퍼파라미터 세팅

In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. 최적화된 배치 사이즈 적용 (64)
batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. 최적화된 노드 수로 Model Class 재선언
class FinalRawInsuranceModel(nn.Module):
    def __init__(self, input_dim):
        super(FinalRawInsuranceModel, self).__init__()
        # Optuna 최적화 노드 수 적용: 44 -> 22 -> 19
        self.linear_1 = nn.Linear(input_dim, 44)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(44, 22)
        self.linear_3 = nn.Linear(22, 19)
        self.linear_4 = nn.Linear(19, 1)

    def forward(self, x):
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.relu(self.linear_3(x))
        return self.linear_4(x)

# 디바이스 설정 및 모델 선언
model = FinalRawInsuranceModel(input_dim).to(device)

# 3. 최적화된 학습률 적용 (0.05414)
optimizer = optim.Adam(params=model.parameters(), lr=0.054140676564044296)
loss_fn = nn.MSELoss().to(device)

#2. 최종 모델 학습 및 평가

In [13]:
epochs = 500

print("🚀 파생 변수(High_Risk) + Optuna 최적화 모델 최종 학습을 시작합니다...")
for epoch in range(epochs):
    model.train()
    train_cost = 0

    for train_inputs, train_labels in train_dataloader:
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        optimizer.zero_grad()
        train_preds = model(train_inputs)
        train_loss = loss_fn(train_preds, train_labels)
        train_loss.backward()
        optimizer.step()

        train_cost += train_loss.item()

    train_cost /= len(train_dataloader)

    # 100 Epoch 단위로 학습 경과 출력
    if (epoch + 1) % 100 == 0:
        print(f"Epoch: {epoch+1}/{epochs} => Train Loss(MSE): {train_cost:.4f}")

# ---------------------------------------------------------
# 최종 Evaluation (로그 변환을 안 했으므로 지수 복원 없이 바로 평가!)
# ---------------------------------------------------------
model.eval()

with torch.no_grad():
    # 테스트 데이터 예측
    test_preds = model(x_test_tensor.to(device)).cpu().numpy()
    test_trues = y_test_tensor.numpy()

# 평가 지표 계산
mse = mean_squared_error(test_trues, test_preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_trues, test_preds)
r2 = r2_score(test_trues, test_preds)

print("\n=== 🏆 파생 변수 추가 & 최종 최적화 모델 평가 결과 ===")
print(f"MSE (평균 제곱 오차): {mse:.2f}")
print(f"RMSE (평균 제곱근 오차): {rmse:.2f}")
print(f"MAE (평균 절대 오차): {mae:.2f}")
print(f"R2 Score (결정계수): {r2:.4f}")

🚀 파생 변수(High_Risk) + Optuna 최적화 모델 최종 학습을 시작합니다...
Epoch: 100/500 => Train Loss(MSE): 18281915.7857
Epoch: 200/500 => Train Loss(MSE): 17138063.9286
Epoch: 300/500 => Train Loss(MSE): 17536708.1429
Epoch: 400/500 => Train Loss(MSE): 17440639.9286
Epoch: 500/500 => Train Loss(MSE): 15425029.1429

=== 🏆 파생 변수 추가 & 최종 최적화 모델 평가 결과 ===
MSE (평균 제곱 오차): 24478784.00
RMSE (평균 제곱근 오차): 4947.60
MAE (평균 절대 오차): 2794.15
R2 Score (결정계수): 0.8363
